<a href="https://colab.research.google.com/github/singhsansu07/Ai-resume-shortlister/blob/main/Ai_resume_shortlister.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================
# CELL 1: INSTALL DEPENDENCIES
# ================================

!pip -q install pymupdf python-docx sentence-transformers gradio pandas numpy scikit-learn openpyxl

print("All dependencies installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.1 MB/s eta 0:00:00
All dependencies installed successfully!


In [2]:
# ================================
# CELL 2: IMPORT LIBRARIES
# ================================

import os
import re
import numpy as np
import pandas as pd
import fitz  # PyMuPDF

from docx import Document
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# ================================
# CELL 3: LOAD NLP MODEL
# ================================

MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("NLP model loaded:", MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

NLP model loaded: all-MiniLM-L6-v2


In [4]:
# ================================
# CELL 4: SKILL DATABASE
# ================================

SKILLS = sorted(set([

    # Programming Languages
    "python",
    "java",
    "c",
    "c++",
    "c#",
    "javascript",
    "typescript",
    "go",
    "rust",
    "php",
    "r",
    "sql",
    "html",
    "css",

    # AI / ML / Data Science
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "nlp",
    "natural language processing",
    "computer vision",
    "data science",
    "data analysis",
    "generative ai",
    "llm",
    "large language models",

    # ML Libraries
    "tensorflow",
    "pytorch",
    "keras",
    "scikit-learn",
    "pandas",
    "numpy",
    "opencv",
    "hugging face",
    "transformers",

    # Web Development
    "react",
    "angular",
    "vue",
    "node.js",
    "express",
    "django",
    "flask",
    "fastapi",
    "rest api",
    "api",
    "microservices",

    # Databases
    "mysql",
    "postgresql",
    "mongodb",
    "sqlite",
    "oracle",
    "redis",

    # Cloud / DevOps
    "aws",
    "azure",
    "google cloud",
    "gcp",
    "docker",
    "kubernetes",
    "git",
    "github",
    "gitlab",
    "linux",
    "ci/cd",

    # Computer Science
    "data structures",
    "algorithms",
    "object oriented programming",
    "oops",
    "operating systems",
    "computer networks",
    "dbms",
    "database management",
    "software engineering",

    # Professional Skills
    "communication",
    "leadership",
    "teamwork",
    "problem solving",
    "project management",
    "agile",
    "scrum"
]))

print("Total skills:", len(SKILLS))

Total skills: 78


In [5]:
# ================================
# CELL 5: TEXT EXTRACTION
# ================================

def extract_pdf_text(file_path):

    text = []

    with fitz.open(file_path) as doc:

        for page in doc:
            text.append(page.get_text("text"))

    return "\n".join(text)


def extract_docx_text(file_path):

    doc = Document(file_path)

    paragraphs = []

    # Normal paragraphs
    for paragraph in doc.paragraphs:
        paragraphs.append(paragraph.text)

    # Tables
    for table in doc.tables:

        for row in table.rows:

            row_text = []

            for cell in row.cells:
                row_text.append(cell.text)

            paragraphs.append(" | ".join(row_text))

    return "\n".join(paragraphs)


def extract_text(file_path):

    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".pdf":

        return extract_pdf_text(file_path)

    elif extension == ".docx":

        return extract_docx_text(file_path)

    elif extension == ".txt":

        with open(
            file_path,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as file:

            return file.read()

    else:

        raise ValueError(
            f"Unsupported file type: {extension}"
        )


def clean_text(text):

    text = text.replace("\x00", " ")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


print("Text extraction functions ready!")

Text extraction functions ready!


In [6]:
# ================================
# CELL 6: RESUME INFORMATION
# ================================

def normalize(text):

    return re.sub(
        r"[^a-z0-9+#.\- ]+",
        " ",
        text.lower()
    )


def extract_email(text):

    pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

    match = re.search(
        pattern,
        text
    )

    if match:
        return match.group(0)

    return ""


def extract_phone(text):

    patterns = [

        r"(?:\+91[-\s]?)?[6-9]\d{9}",

        r"(?:\+\d{1,3}[-\s]?)?\(?\d{3}\)?[-\s.]?\d{3}[-\s.]?\d{4}"
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text
        )

        if match:
            return match.group(0)

    return ""


def extract_skills(text):

    normalized_text = normalize(text)

    found_skills = []

    for skill in SKILLS:

        normalized_skill = normalize(skill).strip()

        if not normalized_skill:
            continue

        pattern = (
            r"(?<!\w)"
            + re.escape(normalized_skill)
            + r"(?!\w)"
        )

        if re.search(
            pattern,
            normalized_text
        ):

            found_skills.append(skill)

    return sorted(set(found_skills))


def extract_name(text, file_path):

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    for line in lines[:8]:

        if (
            2 <= len(line.split()) <= 5
            and len(line) <= 60
            and not re.search(
                r"@|\d{5,}|resume|curriculum|cv",
                line,
                re.I
            )
        ):

            return line

    return os.path.splitext(
        os.path.basename(file_path)
    )[0]


print("Candidate information extraction ready!")

Candidate information extraction ready!


In [7]:
# ================================
# CELL 8: SEMANTIC SIMILARITY
# ================================

def semantic_similarity(
    text_a,
    text_b
):

    embeddings = model.encode(
        [
            text_a,
            text_b
        ],
        normalize_embeddings=True,
        show_progress_bar=False
    )

    score = float(
        np.dot(
            embeddings[0],
            embeddings[1]
        )
    )

    score = max(
        0.0,
        min(
            1.0,
            score
        )
    )

    return score


print("Semantic similarity function ready!")

Semantic similarity function ready!


In [8]:
# ================================
# CELL 10: FINAL SCORE
# ================================

def calculate_final_score(
    semantic_score,
    skill_score_value
):

    final_score = (
        0.70 * semantic_score
        +
        0.30 * skill_score_value
    )

    return round(
        final_score * 100,
        2
    )


print("Scoring system ready!")

Scoring system ready!


In [9]:
# ================================
# CELL 11: MAIN RANKING ENGINE
# ================================

def rank_resumes(
    file_paths,
    job_description
):

    jd = extract_job_requirements(
        job_description
    )

    if not jd["text"]:

        raise ValueError(
            "Job description is empty."
        )

    if not file_paths:

        raise ValueError(
            "No resumes were provided."
        )

    results = []

    for file_path in file_paths:

        try:

            # -------------------------
            # Extract resume text
            # -------------------------

            raw_text = extract_text(
                file_path
            )

            text = clean_text(
                raw_text
            )

            if len(text) < 30:

                raise ValueError(
                    "Very little readable text found."
                )

            # -------------------------
            # Extract skills
            # -------------------------

            resume_skills = extract_skills(
                text
            )

            # -------------------------
            # Semantic similarity
            # -------------------------

            semantic = semantic_similarity(
                text,
                jd["text"]
            )

            # -------------------------
            # Skill matching
            # -------------------------

            skill_coverage, matched, missing = skill_score(
                resume_skills,
                jd["skills"]
            )

            # -------------------------
            # Final score
            # -------------------------

            final_score = calculate_final_score(
                semantic,
                skill_coverage
            )

            # -------------------------
            # Candidate information
            # -------------------------

            candidate_name = extract_name(
                raw_text,
                file_path
            )

            email = extract_email(
                raw_text
            )

            phone = extract_phone(
                raw_text
            )

            # -------------------------
            # Shortlisting
            # -------------------------

            if final_score >= 65:

                status = "Shortlisted"

            else:

                status = "Review"

            results.append({

                "Candidate":
                    candidate_name,

                "Resume File":
                    os.path.basename(file_path),

                "Email":
                    email,

                "Phone":
                    phone,

                "Final Score (%)":
                    final_score,

                "Semantic Match (%)":
                    round(
                        semantic * 100,
                        2
                    ),

                "Skill Match (%)":
                    round(
                        skill_coverage * 100,
                        2
                    ),

                "Matched Skills":
                    ", ".join(matched),

                "Missing Skills":
                    ", ".join(missing),

                "Resume Skills":
                    ", ".join(resume_skills),

                "Status":
                    status
            })

        except Exception as error:

            results.append({

                "Candidate":
                    os.path.splitext(
                        os.path.basename(file_path)
                    )[0],

                "Resume File":
                    os.path.basename(file_path),

                "Email":
                    "",

                "Phone":
                    "",

                "Final Score (%)":
                    0,

                "Semantic Match (%)":
                    0,

                "Skill Match (%)":
                    0,

                "Matched Skills":
                    "",

                "Missing Skills":
                    "",

                "Resume Skills":
                    "",

                "Status":
                    f"Error: {str(error)}"
            })

    # -------------------------
    # Create DataFrame
    # -------------------------

    df = pd.DataFrame(
        results
    )

    # -------------------------
    # Sort candidates
    # -------------------------

    df = df.sort_values(
        "Final Score (%)",
        ascending=False
    )

    df = df.reset_index(
        drop=True
    )

    # -------------------------
    # Add Rank
    # -------------------------

    df.insert(
        0,
        "Rank",
        range(
            1,
            len(df) + 1
        )
    )

    return df


print("Resume ranking engine ready!")

Resume ranking engine ready!


In [10]:
# ================================
# CELL 12: UPLOAD RESUMES
# ================================

from google.colab import files

uploaded = files.upload()

resume_files = list(
    uploaded.keys()
)

print(
    "Uploaded resumes:"
)

for file in resume_files:

    print(
        "-",
        file
    )

Saving UpdatedResumeDataSet.csv to UpdatedResumeDataSet.csv
Uploaded resumes:
- UpdatedResumeDataSet.csv


In [11]:
# ================================
# CELL 13: JOB DESCRIPTION
# ================================

job_description = """

We are hiring a Python AI/ML Developer.

Responsibilities:

Develop machine learning applications.
Build NLP based applications.
Work with Python and SQL.
Develop data processing pipelines.
Work with REST APIs.
Use Git and GitHub for version control.

Required Skills:

Python
Machine Learning
Artificial Intelligence
NLP
SQL
Pandas
NumPy
Scikit-learn
Git
GitHub

Preferred Skills:

FastAPI
Docker
AWS
Deep Learning
TensorFlow

Good communication and problem solving skills are required.

"""

In [14]:
# ================================
# CELL 14: RUN RESUME SHORTLISTER
# ================================

ranking = rank_resumes(
    resume_files,
    job_description
)

display(
    ranking
)

,Rank,Candidate,Resume File,Email,Phone,Final Score (%),Semantic Match (%),Skill Match (%),Matched Skills,Missing Skills,Resume Skills,Status
0,1,UpdatedResumeDataSet,UpdatedResumeDataSet.csv,,,0,0,0,,,,Error: Unsupported file type: .csv


In [13]:
# ================================
# CELL 9: JOB DESCRIPTION PROCESSING
# ================================

def extract_job_requirements(job_description):
    jd_skills = extract_skills(job_description)

    return {
        "text": clean_text(job_description),
        "skills": jd_skills
    }


def skill_score(resume_skills, job_skills):
    if not job_skills:
        return 0.0, [], []

    matched = [skill for skill in resume_skills if skill in job_skills]
    missing = [skill for skill in job_skills if skill not in resume_skills]

    score = len(matched) / len(job_skills)

    return score, matched, missing

print("Job description processing functions ready!")

Job description processing functions ready!


In [15]:
# ================================
# CELL 15: EXPORT CSV
# ================================

output_file = (
    "/content/resume_shortlist.csv"
)

ranking.to_csv(
    output_file,
    index=False
)

print(
    "Shortlist saved at:",
    output_file
)

Shortlist saved at: /content/resume_shortlist.csv


In [16]:
# ================================
# CELL 17: GRADIO WEB APP
# ================================

import gradio as gr


def run_shortlisting(
    job_description,
    resume_files
):

    if not job_description:

        return (
            pd.DataFrame({
                "Error": [
                    "Please enter a job description."
                ]
            }),
            None
        )

    if not resume_files:

        return (
            pd.DataFrame({
                "Error": [
                    "Please upload resumes."
                ]
            }),
            None
        )

    # -------------------------
    # Get uploaded paths
    # -------------------------

    paths = []

    for file in resume_files:

        if hasattr(
            file,
            "name"
        ):

            paths.append(
                file.name
            )

        else:

            paths.append(
                str(file)
            )

    try:

        # -------------------------
        # Rank resumes
        # -------------------------

        df = rank_resumes(
            paths,
            job_description
        )

        # -------------------------
        # Save CSV
        # -------------------------

        output_path = (
            "/content/resume_shortlist.csv"
        )

        df.to_csv(
            output_path,
            index=False
        )

        return (
            df,
            output_path
        )

    except Exception as error:

        return (
            pd.DataFrame({
                "Error": [
                    str(error)
                ]
            }),
            None
        )


demo = gr.Interface(

    fn=run_shortlisting,

    inputs=[

        gr.Textbox(
            label="Job Description",
            placeholder=(
                "Paste the job description here..."
            ),
            lines=12
        ),

        gr.File(
            label="Upload Resumes",
            file_count="multiple",
            file_types=[
                ".pdf",
                ".docx",
                ".txt"
            ],
            type="filepath"
        )
    ],

    outputs=[

        gr.Dataframe(
            label="Ranked Candidates",
            interactive=False
        ),

        gr.File(
            label="Download Shortlist CSV"
        )
    ],

    title="AI Resume Shortlister",

    description=(
        "Upload multiple resumes and paste "
        "a job description. The AI system "
        "will calculate semantic relevance, "
        "skill coverage and rank candidates."
    ),

    flagging_mode="never"
)


demo.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8df77fa60ac4345926.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
# ================================
# CELL 16: DOWNLOAD CSV
# ================================

from google.colab import files

files.download(
    "/content/resume_shortlist.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# ================================
# CELL 17: GRADIO WEB APP
# ================================

import gradio as gr


def run_shortlisting(
    job_description,
    resume_files
):

    if not job_description:

        return (
            pd.DataFrame({
                "Error": [
                    "Please enter a job description."
                ]
            }),
            None
        )

    if not resume_files:

        return (
            pd.DataFrame({
                "Error": [
                    "Please upload resumes."
                ]
            }),
            None
        )

    # -------------------------
    # Get uploaded paths
    # -------------------------

    paths = []

    for file in resume_files:

        if hasattr(
            file,
            "name"
        ):

            paths.append(
                file.name
            )

        else:

            paths.append(
                str(file)
            )

    try:

        # -------------------------
        # Rank resumes
        # -------------------------

        df = rank_resumes(
            paths,
            job_description
        )

        # -------------------------
        # Save CSV
        # -------------------------

        output_path = (
            "/content/resume_shortlist.csv"
        )

        df.to_csv(
            output_path,
            index=False
        )

        return (
            df,
            output_path
        )

    except Exception as error:

        return (
            pd.DataFrame({
                "Error": [
                    str(error)
                ]
            }),
            None
        )


demo = gr.Interface(

    fn=run_shortlisting,

    inputs=[

        gr.Textbox(
            label="Job Description",
            placeholder=(
                "Paste the job description here..."
            ),
            lines=12
        ),

        gr.File(
            label="Upload Resumes",
            file_count="multiple",
            file_types=[
                ".pdf",
                ".docx",
                ".txt"
            ],
            type="filepath"
        )
    ],

    outputs=[

        gr.Dataframe(
            label="Ranked Candidates",
            interactive=False
        ),

        gr.File(
            label="Download Shortlist CSV"
        )
    ],

    title="AI Resume Shortlister",

    description=(
        "Upload multiple resumes and paste "
        "a job description. The AI system "
        "will calculate semantic relevance, "
        "skill coverage and rank candidates."
    ),

    flagging_mode="never"
)


demo.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d574ef862128b2e926.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
